# SQL trainging ground

The goal of this notebook is to practice SQL. I'm going to answer some questions from easy to complex. 

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../src')

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np 

from loaders import *

import sqlite3

db_path = "../data/station_data.db"
connection = sqlite3.connect(db_path)
cursor = connection.cursor()

In [ ]:
df = load_range(2000,1,2023,2)
df.head()

In [ ]:
df.to_sql("weather", con = connection, if_exists="replace", index=False, chunksize = 10_000)

The table is created.

In [ ]:
cursor.execute("SELECT COUNT(*) FROM weather;")
cursor.fetchone()

## Answering questions with SQL

### Selecting the first 5 datapoints

In [ ]:
query = """SELECT * FROM weather LIMIT 5;"""

pd.read_sql_query(query,connection)

### What data was recorded on january 1, 2000?

In [ ]:
query = """SELECT * FROM weather WHERE date<'2000-01-02' AND date>='2000-01-01'"""
pd.read_sql_query(query,connection)

### What was the average temperature on January 1, 2000?
Answer: 2.922 ºC

In [ ]:
query = """SELECT AVG(temp) FROM weather WHERE date<'2000-01-02' AND date>='2000-01-01'"""

pd.read_sql_query(query, connection)

### What were the minimim and maximum temperatures on March 21, 2007? (firend's bithday)

In [ ]:
query = """SELECT MAX(temp) AS max_temp ,MIN(temp) AS min_temp FROM weather WHERE date<'2007-03-22' AND date>='2007-03-21'"""

result = pd.read_sql_query(query, connection)

result.style.hide(axis="index")

In [ ]:
query = """SELECT MAX(temp) AS max_temp ,MIN(temp) AS min_temp FROM weather WHERE date>='2010-07-02' AND date<'2010-07-03'"""

result = pd.read_sql_query(query, connection)

result.style.hide(axis="index")

### How did temperature change on November 28, 2007? (friend's birthday)

In [ ]:
query = """SELECT strftime('%H:%M',date) AS time,temp FROM weather WHERE date>='2007-11-28' AND date<'2007-11-29'"""

temp_daniel = pd.read_sql_query(query, connection)

temp_daniel.style.hide(axis="index")

In [ ]:
plt.figure(dpi = 200)
plt.plot(temp_daniel["time"],temp_daniel["temp"], color = "red")
plt.xticks(temp_daniel["time"][::12],rotation = 45)
plt.axvline(x = "06:00", color = "grey", label = "Daniel", linestyle = ":")
plt.axvspan("00:00", "06:00", color="gray", alpha=0.12)


plt.xlabel("Hora")
plt.ylabel("Temperatura ºC")
plt.title("Temperatura el 28 Nov 2007 - Salamanca")
plt.show()

### How many observations were recorded each year? 

In [ ]:
query = """SELECT strftime('%Y',date) AS year, COUNT(*) AS number_of_observations FROM weather GROUP BY year ORDER BY year"""

result = pd.read_sql_query(query, connection)

result.style.hide(axis="index")

### What was the average temperature for each year?

In [ ]:
query = """SELECT strftime('%Y',date) AS year, AVG(temp) AS avg_temp FROM weather WHERE date < '2023-01-01' GROUP BY year ORDER BY year"""

result = pd.read_sql_query(query, connection)

result.style.hide(axis="index")

In [ ]:
plt.figure(dpi = 200)

plt.plot(result["year"], result["avg_temp"], color = "red")
plt.xticks(rotation = 45)
plt.xlabel("year")
plt.ylabel("avg_temp ºC")
plt.title("Average temperature 2000-2022")
plt.grid()
plt.tight_layout()

### How did missing temperature values change by year?

In [ ]:
query = """SELECT strftime('%Y',date) AS year, COUNT(*)-COUNT(temp) AS missing_temp FROM weather GROUP BY year ORDER BY year"""

result = pd.read_sql_query(query, connection)

result.style.hide(axis="index")

### Which year was the hottest and coldest in terms of yearly average temperature?

In [ ]:
query = """ SELECT * FROM (SELECT strftime('%Y',date) AS year, AVG(temp) as avg_temp FROM weather WHERE strftime('%Y',date) != '2023' GROUP BY year ORDER BY avg_temp ASC LIMIT 1)
UNION ALL
SELECT * FROM (SELECT strftime('%Y',date) AS year, AVG(temp) as avg_temp FROM weather WHERE strftime('%Y',date) != '2023' GROUP BY year ORDER BY avg_temp DESC LIMIT 1) 
;"""

pd.read_sql_query(query,connection)

### Are there specific years with worse data quality than others?

In [ ]:
query = """SELECT strftime('%Y',date) AS year, ROUND( 100.0* (COUNT(*)-COUNT(temp))/COUNT(*),2) AS missing_temp FROM weather GROUP BY year ORDER BY year"""

result = pd.read_sql_query(query, connection)

result.style.hide(axis="index")

### For each month, what is the average temp and gap between the hottest and coldest year for that month? 

In [ ]:
query = """WITH monthly_yearly_avg AS (
SELECT 
    strftime('%m', date) AS month,
    strftime('%Y', date) AS year,
    AVG(temp) AS avg_temp 
    FROM weather 
    GROUP BY month, year
)
SELECT
    month, 
    AVG(avg_temp) AS overall_avg,
    MAX(avg_temp) - MIN(avg_temp) AS gap
FROM monthly_yearly_avg
GROUP BY month;
"""


result = pd.read_sql_query(query, connection)

result.style.hide(axis="index")

### Which years had a streak of more than 15 days without raining?

I need to create a table with each day and the total rain measured that day, all in one row.

In [ ]:
query = """
SELECT
    DATE(date) AS day, 
    SUM(rain_mm) AS total_rain
FROM weather 
GROUP BY DATE(date)
"""

# the date function removes specific times because it searches for YYYY-MM-DD format 

result = pd.read_sql_query(query, connection)
print(result.head(10))

In [ ]:
query = """
WITH daily_rain AS (
    SELECT
        DATE(date) AS day,
        SUM(rain_mm) AS total_rain
    FROM weather 
    GROUP BY DATE(date)
)
SELECT
    day, 
    ROW_NUMBER() OVER (ORDER BY day) AS row_num,
    julianday(day) - ROW_NUMBER() OVER (ORDER BY day) AS streak_id
FROM daily_rain
WHERE total_rain = 0
"""


 #  ROW_NUMBER() OVER (ORDER BY day) AS row_num, is a window function that assigns a number 1, 2, 3 ... to a row following day order
 # julianday(day) assings a decimal number, this is the number of days since 12:00 on 4713 aC
 # julianday(day)-ROW_NUMBER() OVER (ORDER BY day) AS streak_id creates an id for each row. this is because if day and row number grow by 1 (consecutive days),
 # streak_id will have the same value during every day in that streak

result = pd.read_sql_query(query, connection)
print(result.head(20))

In [ ]:
query = """
WITH 

daily_rain AS (
    SELECT
        DATE(date) AS day,
        SUM(rain_mm) AS total_rain
    FROM weather 
    GROUP BY DATE(date)
),
dry_days AS (
SELECT
    day, 
    ROW_NUMBER() OVER (ORDER BY day) AS row_num,
    julianday(day) - ROW_NUMBER() OVER (ORDER BY day) AS streak_id
FROM daily_rain
WHERE total_rain = 0
),
qualifying_streaks AS (
    SELECT
        -- streak_id,
        strftime('%Y', MIN(day)) AS year

        -- COUNT(*) AS streak_length, 
        -- MIN(day) AS streak_start,
        -- MAX(day) AS streak_end
    FROM dry_days
    GROUP BY streak_id
    HAVING COUNT(*) >=15
    ORDER BY COUNT(*) DESC
)
SELECT year,  COUNT(*) AS number_of_longer_than_15_days_streaks
    FROM qualifying_streaks
    GROUP BY year
;
"""


 #  ROW_NUMBER() OVER (ORDER BY day) AS row_num, is a window function that assigns a number 1, 2, 3 ... to a row following day order
 # julianday(day) assings a decimal number, this is the number of days since 12:00 on 4713 aC
 # julianday(day)-ROW_NUMBER() OVER (ORDER BY day) AS streak_id creates an id for each row. this is because if day and row number grow by 1 (consecutive days),
 # streak_id will have the same value during every day in that streak
 # then we select streak id and compute the streak lenght and where did it start and end

result = pd.read_sql_query(query, connection)
print(result.head(20))

### What were the days with the most thermal amplitude? 

CTE means common table expression

In [ ]:
query = """ 
WITH thermal_amplitude AS (
    
    SELECT 
        DATE(date) AS day,
        MAX(temp) AS max_temp,
        MIN(temp) AS min_temp
    FROM weather
    GROUP BY DATE(date)
)

SELECT *,(max_temp  - min_temp) AS temp_diff FROM thermal_amplitude ORDER BY (max_temp  - min_temp) DESC LIMIT 10;

"""

result = pd.read_sql_query(query, connection)
print(result.head(20))

### Calculate the mean temperature for each available decade. 

In [ ]:
query = """
SELECT strftime('%Y',date) AS year, AVG(temp) FROM weather  GROUP BY year HAVING strftime('%Y',date) != '2023' ORDER BY year DESC;
"""

result = pd.read_sql_query(query,connection)
result

### What was the hottest day each year? 

In [ ]:
query = """
WITH hottest_days AS (
    SELECT 
        strftime('%Y', date) AS year, 
        MAX(temp) AS max_temp
    FROM weather
    GROUP BY strftime('%Y', date)
)
SELECT w.*
FROM weather w
JOIN hottest_days h
    ON strftime('%Y', w.date) = h.year AND w.temp = h.max_temp
"""

result = pd.read_sql_query(query, connection)
result

### What was the largest temperature jump between two days?

In [ ]:
query = """ 
WITH averages AS (
SELECT 
DATE(date) AS date,
AVG(temp) AS avg_temp
FROM weather
GROUP BY DATE(date)
),
comparator AS (
SELECT date, avg_temp, LAG(avg_temp) OVER (ORDER BY date) AS previous_avg_temp
FROM averages
),
jumps AS (
SELECT date, ABS(avg_temp - previous_avg_temp) AS jump
FROM comparator 
where previous_avg_temp IS NOT NULL
),
ranked AS (
    SELECT date, jump,
        ROW_NUMBER() OVER (ORDER BY jump DESC) AS rn
    FROM jumps
)
SELECT date, jump 
FROM ranked
WHERE rn = 1
;
"""


result = pd.read_sql_query(query, connection)
result


### Calculate a 7-day moving average and indnetify the 5 days when the temperature grew apart the most with the average. 

In [ ]:
query = """ 
WITH moving_average AS (
SELECT DATE(date) AS day, AVG(temp) OVER (ORDER BY DATE(date) ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS avg_seven_days,
AVG(temp) AS avg_temp 
FROM weather
GROUP BY day
),
differences AS (
    SELECT day, avg_temp, avg_seven_days, ABS(avg_temp - avg_seven_days) AS difference
    FROM moving_average 
),
ranked AS (
    SELECT day, difference,
        ROW_NUMBER() OVER (ORDER BY difference DESC) AS rn
    FROM differences
)
SELECT day, difference 
FROM ranked
WHERE rn <= 5
;
"""


result = pd.read_sql_query(query, connection)
result

### 7 day centered moving average

In [ ]:
query = """ 
WITH day_average AS (
SELECT DATE(date) AS day, AVG(temp) as daily_average
FROM weather 
GROUP BY DATE(date)
),
moving_average AS (
SELECT day, AVG(daily_average) OVER (ORDER BY day ROWS BETWEEN 3 PRECEDING AND 3 FOLLOWING) AS centered_avg_seven_days
FROM day_average
)
SELECT day, centered_avg_seven_days 
FROM moving_average
;
"""


result = pd.read_sql_query(query, connection)
result

### For each year, calculate the number of "heat waves" defined as 3+ consecutive days where max temperaature was above 95 percentile. 

In [ ]:
query = """ 

WITH percentiles AS (
SELECT -- colapses every day to its date and calculates percentile
DATE(date) AS day,
PERCENT_RANK() OVER (PARTITION BY strftime('%Y',date) ORDER BY MAX(temp)) AS percentile
FROM weather
GROUP BY day
),
flagged_days AS ( -- creates column with 0 or 1 depending if it's a really hot day
SELECT day, percentile, CASE WHEN percentile >= 0.95 THEN 1 ELSE 0 END AS is_hot_day
FROM percentiles
ORDER BY day ASC
), 
target_days AS ( -- calculates a streak id with the julianday and the row number only considering flagged days
SELECT 
    day,
    ROW_NUMBER() OVER (ORDER BY day) AS row_num,
    julianday(day) - ROW_NUMBER() OVER (ORDER BY day) AS streak_id
FROM flagged_days
WHERE is_hot_day = 1
), -- qualifying streaks are grouped by streak id and the mimnimum day is used to calculate the year. also, >=3 streaks are filtered
qualifying_streaks AS (
    SELECT 
    strftime('%Y',MIN(day)) AS year, streak_id
FROM target_days
GROUP BY streak_id
HAVING COUNT(*) >=3
)
SELECT year, COUNT(*) FROM qualifying_streaks
GROUP BY year
;
"""

result = pd.read_sql_query(query, connection)
print(result)